# Model Context Protocol (MCP)

[Model Context Protocol (MCP)](https://modelcontextprotocol.io) là một giao thức mở nhằm chuẩn hóa cách các ứng dụng cung cấp tool và context cho các mô hình ngôn ngữ. Các agent LangChain gọi các tool được định nghĩa trên các MCP server thông qua [`MCPAdapter`](https://reference.langchain.com/python/langchain/mcp/adapter/MCPAdapter), lớp này sẽ khám phá các tool của một server và chuyển đổi chúng thành các tool LangChain mà bạn có thể truyền thẳng vào [`create_agent`](https://reference.langchain.com/python/langchain/agents/factory/create_agent).

[`MCPAdapter`](https://reference.langchain.com/python/langchain/mcp/adapter/MCPAdapter) được xây dựng trên nền tảng [FastMCP](https://gofastmcp.com), thành phần chịu trách nhiệm xử lý việc suy luận transport, đàm phán giao thức, quản lý kết nối và xác thực. Phần này đề cập đến lớp riêng của LangChain và liên kết đến tài liệu FastMCP client để tìm hiểu chi tiết về kết nối bên dưới.

<div class="alert alert-info">

Namespace `langchain.mcp` yêu cầu `langchain[mcp]>=1.4.0` và đang ở giai đoạn beta. Việc import từ namespace này sẽ phát sinh cảnh báo `LangChainBetaWarning` một lần cho mỗi tiến trình. API có thể sẽ còn thay đổi.

Nếu bạn đã sử dụng MCP trước phiên bản v1.4.0, hãy xem [Chuyển đổi từ `langchain-mcp-adapters`](https://docs.langchain.com/oss/python/migrate/langchain-mcp-adapters).

</div>

## Cài đặt

Cài đặt LangChain với extra `mcp`, thành phần này sẽ kéo theo FastMCP:

**pip**

```
pip install "langchain[mcp]"
```

**uv**

```
uv add "langchain[mcp]"
```

## Bắt đầu nhanh

Mở một [`MCPAdapter`](https://reference.langchain.com/python/langchain/mcp/adapter/MCPAdapter), khám phá các tool của server bằng `list_tools()`, và xây dựng agent bên trong context. Các tool nắm giữ client, vì vậy agent vẫn có thể sử dụng được sau khi context kết thúc:

In [ ]:
from langchain.agents import create_agent
from langchain.mcp import MCPAdapter


async def main():
    async with MCPAdapter("https://example.com/mcp") as adapter:
        tools = await adapter.list_tools()
        agent = create_agent("claude-sonnet-5", tools)
        return await agent.ainvoke({"messages": [{"role": "user", "content": "..."}]})

**Ví dụ: Truy vấn tài liệu LangChain**
  
[MCP server tài liệu LangChain](https://docs.langchain.com/use-these-docs) là một endpoint HTTP công khai tại `https://docs.langchain.com/mcp`. Kết nối một agent với server này để tìm kiếm và đọc tài liệu mà không cần viết tool tùy chỉnh:

In [ ]:
from langchain.agents import create_agent
from langchain.mcp import MCPAdapter


async def main():
    async with MCPAdapter("https://docs.langchain.com/mcp") as adapter:
        tools = await adapter.list_tools()
        agent = create_agent("claude-sonnet-5", tools)
        return await agent.ainvoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": "Làm thế nào để thêm bộ nhớ ngắn hạn (short-term memory) cho một agent LangChain?",
                    }
                ]
            }
        )

<div class="alert alert-info">

Server MCP tài liệu là công khai và không yêu cầu API key. Để biết cách thiết lập cho IDE và coding agent (Claude Code, Cursor, và các công cụ khác), hãy xem [Sử dụng tài liệu theo cách lập trình](https://docs.langchain.com/use-these-docs).

</div>

Server này cung cấp các tool sau:

| Tool                                       | Mô tả                                                                                   |
| ------------------------------------------ | --------------------------------------------------------------------------------------------- |
| `search_docs_by_lang_chain`                | Tìm kiếm tài liệu để tìm các hướng dẫn, cách thực hiện và ví dụ liên quan.                                       |
| `query_docs_filesystem_docs_by_lang_chain` | Đọc hoặc tìm kiếm tài liệu thông qua một hệ thống tệp ảo (sử dụng các lệnh `rg`, `head`, `cat`, và các lệnh liên quan). |
| `submit_feedback`                          | Báo cáo sự cố với một trang tài liệu.                                                   |

## Transport

[`MCPAdapter`](https://reference.langchain.com/python/langchain/mcp/adapter/MCPAdapter) suy luận transport từ target mà bạn truyền vào, vì vậy điều duy nhất thay đổi giữa một server chạy in-process, một script local qua stdio, và một URL từ xa chính là target đó:

In [ ]:
from pathlib import Path

from langchain.mcp import MCPAdapter

# Một FastMCP server chạy in-process: không có subprocess, không có socket. Lý tưởng cho việc kiểm thử (test).
in_memory = MCPAdapter(server)  # một instance FastMCP

# Một script path sẽ được khởi chạy qua stdio, mỗi adapter tương ứng với một subprocess.
stdio = MCPAdapter(Path("weather_server.py"))

# Một chuỗi (string) phải là một URL http(s), được kết nối qua streamable HTTP.
http = MCPAdapter("https://example.com/mcp")

Một target có thể là bất kỳ loại nào sau đây:

* **Một URL `http`/`https`** (`str`): được kết nối qua streamable HTTP.
* **Một script path** (`Path`): được khởi chạy dưới dạng subprocess qua stdio.
* **Một đối tượng transport** (`StreamableTransport`): một đối tượng transport đã được cấu hình sẵn. Xem [Client Transports](https://gofastmcp.com/clients/transports).
* **Một `FastMCP` server chạy in-process**: được kết nối in-memory, không có subprocess hay socket.
* **Một dict `MCPConfig`** (`{"mcpServers": {...}}`): nhiều server đứng sau một adapter. Xem [Connections](https://docs.langchain.com/oss/python/langchain/mcp/connections#multiple-servers).
* **Một `fastmcp.Client` đã được xây dựng sẵn**: để kiểm soát toàn diện transport, [caching](https://gofastmcp.com/clients/client#response-caching), và [đàm phán giao thức](https://gofastmcp.com/clients/client#protocol-negotiation).

<div class="alert alert-warning">

Một target dạng `str` phải là một URL `http` hoặc `https`. FastMCP sẽ phân giải một chuỗi bằng cách kiểm tra xem nó có phải là một đường dẫn hệ thống tệp (filesystem path) trước, sau đó mới kiểm tra xem nó có phải là một URL, vì vậy một chuỗi trùng tên với một tệp `.py` hoặc `.js` đang tồn tại sẽ khởi chạy tệp đó dưới dạng subprocess. Vì các chuỗi thường là dạng mà target xuất hiện khi đến từ cấu hình hoặc từ một mô hình, [`MCPAdapter`](https://reference.langchain.com/python/langchain/mcp/adapter/MCPAdapter) sẽ từ chối các chuỗi không có định dạng của một URL.

</div>

## Bước tiếp theo

- [**Connections**](https://docs.langchain.com/oss/python/langchain/mcp/connections): Vòng đời kết nối, nhiều server, các thời kỳ giao thức (protocol era), và caching.
- [**Authentication**](https://docs.langchain.com/oss/python/langchain/mcp/auth): Bearer token, OAuth 2.1, và xác thực server theo từng người dùng.
- [**Tools**](https://docs.langchain.com/oss/python/langchain/mcp/tools): Nạp các tool MCP vào agent, kiểm soát việc thực thi chúng, và xử lý đầu ra của chúng.